# Qwen3.5-4B → OpenVINO GenAI (NPU-Ready) + HuggingFace Push

Produces a **single stateful OpenVINO model** ready for `openvino_genai.LLMPipeline` on NPU.

| Step | Action |
|------|--------|
| 0 | Load Kaggle secrets |
| 1 | `git clone dsainvg/openvino-model-conv` |
| 2 | Install requirements |
| 3 | `pytest qwen35/tests/` — toy smoke tests |
| 4 | `download_model.py` — pull Qwen3.5-4B weights |
| 5 | `convert_to_openvino_genai.py` — INT4 stateful single model |
| 5b | Sanity checks — verify files + model graph structure |
| 6 | `push_to_hf.py` — upload to HuggingFace |

> **Kaggle Secrets needed**:
> - `HF_TOKEN` — HuggingFace write token
> - `HF_REPO_NAME` — target repo *(default: `qwen35-4b-openvino-genai-npu`)*

---
**Output**: Split-IR models (`embed_tokens.xml`, `layer_0.xml`, etc.) + `openvino_tokenizer.xml` + `generation_config.json`

**Usage on your NPU machine** (after pulling from HF):
```python
from qwen_npu_pipeline import QwenPipeline
pipe = QwenPipeline('/path/to/model')
print(pipe.generate('Hello, my name is', max_new_tokens=64))
```

## 0 · Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(key, fallback=None):
        try:
            return _s.get_secret(key)
        except Exception:
            return fallback
except ImportError:
    def _get(key, fallback=None):
        return os.environ.get(key, fallback)

HF_TOKEN     = _get('HF_TOKEN')
HF_REPO_NAME = _get('HF_REPO_NAME', 'qwen35-4b-openvino-genai-npu')

if not HF_TOKEN:
    raise EnvironmentError('HF_TOKEN secret is missing. Add it under Add-ons -> Secrets.')

os.environ['HF_TOKEN']               = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print(f'HF_REPO_NAME : {HF_REPO_NAME}')
print('HF_TOKEN     : *** (set)')

## 1 · Clone repo

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/dsainvg/openvino-model-conv.git'
REPO_DIR = Path('/kaggle/working/openvino-model-conv')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, str(REPO_DIR)], check=True)

QWEN35_DIR  = REPO_DIR / 'qwen35'
SCRIPTS_DIR = QWEN35_DIR / 'scripts'
MODEL_DIR   = Path('/kaggle/working/Qwen3.5-4B')
OUTPUT_DIR  = Path('/kaggle/working/ov_genai_qwen35_int4')

print(f'Repo   : {REPO_DIR}')
print(f'Output : {OUTPUT_DIR}')

## 2 · Install requirements

In [ ]:
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('torch', '--index-url', 'https://download.pytorch.org/whl/cpu')
pip(
    'openvino',              # ov.convert_model, ov.make_stateful, ov.save_model
    'nncf',                  # INT4_SYM weight compression
    'openvino-tokenizers',   # convert HF tokenizer to openvino_tokenizer.xml
    'transformers',          # AutoTokenizer (for tokenizer export)
    'huggingface_hub',
    'safetensors',
    'sentencepiece',
    'tiktoken',
    'accelerate',
    'pytest',
)
print('Done.')

## 3 · Toy smoke tests

12 tests — no model download needed. Should finish in under 60 seconds.

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError('Toy smoke tests FAILED — fix code before converting real weights.')
print('\n All tests passed.')

## 4 · Download Qwen3.5-4B weights

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / 'download_model.py'),
        '--model',  'Qwen/Qwen3.5-4B',
        '--output', str(MODEL_DIR),
        '--token',  HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError('download_model.py failed.')
print('\n Model downloaded to', MODEL_DIR)

## 5 · Convert to split-IR OpenVINO model

1. Loads weights layer-by-layer
2. `ov.convert_model()` — traces each layer individually into separate OV graphs
3. `nncf.compress_weights(INT4_SYM, group_size=128)` — compress Linear weights per layer
4. Exports `openvino_tokenizer.xml` + `generation_config.json`

**Expected size**: ~2–2.5 GB &nbsp;&nbsp; **Expected runtime**: ~30 min on Kaggle CPU

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / 'convert_to_openvino.py'),
        '--model-dir', str(MODEL_DIR),
        '--output',    str(OUTPUT_DIR),
        '--dtype',     'fp16',
        '--group-size', '128',
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError('convert_to_openvino.py failed.')
print('\n Split-IR model saved to', OUTPUT_DIR)

## 5b · Sanity checks

Verifies the output **without running inference**:
- All expected files exist and are non-empty
- `generation_config.json` has the right EOS token
- Tokenizer files are valid OV models

In [ ]:
import json
import openvino as ov

REQUIRED_FILES = [
    'embed_tokens.xml',
    'embed_tokens.bin',
    'layer_0.xml',
    'layer_0.bin',
    'openvino_tokenizer.xml',
    'openvino_detokenizer.xml',
    'generation_config.json',
    'config.json',
]

print('--- File checks ---')
all_ok = True
for fname in REQUIRED_FILES:
    fpath = OUTPUT_DIR / fname
    if not fpath.exists():
        print(f'  MISSING : {fname}')
        all_ok = False
    else:
        mb = fpath.stat().st_size / 1e6
        print(f'  OK  {fname:45s} {mb:7.1f} MB')

print('\n--- generation_config.json ---')
with open(OUTPUT_DIR / 'generation_config.json') as f:
    gen_cfg = json.load(f)
print(f'  eos_token_id   : {gen_cfg.get("eos_token_id")}')
print(f'  max_new_tokens : {gen_cfg.get("max_new_tokens")}')

print('\n--- Tokenizer check ---')
core = ov.Core()
try:
    tok_model = core.read_model(str(OUTPUT_DIR / 'openvino_tokenizer.xml'))
    print(f'  OK  openvino_tokenizer has {len(tok_model.inputs)} input(s), {len(tok_model.outputs)} output(s)')
except Exception as e:
    print(f'  WARN  could not read openvino_tokenizer.xml: {e}')

print()
if all_ok:
    print('All sanity checks passed.')
else:
    raise RuntimeError('One or more sanity checks FAILED. See output above.')

## 6 · Push to HuggingFace

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / 'push_to_hf.py'),
        '--ir-dir',    str(OUTPUT_DIR),
        '--repo-name', HF_REPO_NAME,
        '--token',     HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError('push_to_hf.py failed.')
print('\n Uploaded to HuggingFace:', HF_REPO_NAME)